# 14 Text Classification

In [ ]:
import sys

if "google.colab" in sys.modules:
    print("Running in Colab!")
    !pip install unidecode

In [ ]:
import os
import shutil
import string
import random
import pathlib
import unidecode
import collections
import regex as re

import numpy as np
import tensorflow as tf

import matplotlib.pyplot as plt
from IPython.display import clear_output

from IPython.core.magic import register_cell_magic

os.environ["KERAS_BACKEND"] = "jax"

import keras

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

In [ ]:
def plot_history(history):
    accuracy = history.history["accuracy"]
    val_accuracy = history.history["val_accuracy"]
    epochs = range(1, len(accuracy) + 1)
    
    plt.plot(epochs, accuracy, "r--", label="Training accuracy")
    plt.plot(epochs, val_accuracy, "b", label="Validation accuracy")
    plt.title("Training and validation accuracy")
    plt.legend()
    plt.show()

---

## A brief history of natural language processing

The **big** story of recent years has been a move away from **hand-crafted features**.

Well until the 1990s the industry employed many linguists to implement specialist knowledge into rules for NLP systems.

Then, statistical approaches took over, especially with the rise of deep learning.

We leave aside the question of exactly **what** current systems understand from language.

What we know is that we can now train them to take in pieces of text, execute task on them.

- "What’s the topic of this text?" (**text classification**)  
- "Does this text contain abuse?" (**content filtering**)  
- "Does this text sound positive or negative?" (**sentiment analysis**)  
- "What should be the next word in this incomplete sentence?" (**language modeling**)  
- "How would you say this in German?" (**translation**)  
- "How would you summarize this article in one paragraph?" (**summarization**)


## Preparing text data

### Overview

- **standardisation**: make your text easier to process
- **splitting** (**tokenisation**): split into chunks (= tokens)
- **indexing**: turn chunks into numbers (indices in a fixed *vocabulary*)

#### Note: Text encoding

The question of **how language is represented** in computers is an important topic: if you're interested in working with language, do go and learn this!

The [**The Unicode Standard**](https://en.wikipedia.org/wiki/Unicode) is a world-wide consensus to associate characters of all languages with a unique identifier (a number), called a **code point**.

The Python documentation on [Unicode](https://docs.python.org/3/howto/unicode.html).

**Code points** can then be **encoded** in bits ([**UTF-8**](https://en.wikipedia.org/wiki/UTF-8), 8 bits, is one of the ways you can do that, and is backward-compatible with [**ASCII**](https://en.wikipedia.org/wiki/ASCII), 7 bits).

The reference has a very nice video summary on UTF-8.

### Text standardization

- Convert to **lower case**;  
- Remove **punctuation**;
- Convert **special characters** to a standard form (é → e);
- *Stemming*: turn **inflected forms** into base ones ("stares"/"stared"/"staring" → "[stare]")

In [ ]:
s = "I didn't wake up this morning. How did that happen?"

In [ ]:
s.lower()

In [ ]:
# removing punctuation
"".join(char for char in s.lower() if char not in string.punctuation)

In [ ]:
def str_to_ints(s):
    # encode into bytes, then turn into a list
    s_enc = s.encode()
    print(f"{s} | {s_enc} | {list(s_enc)}")
s = "Jérémie"
str_to_ints(s)
str_to_ints(s.lower())
# removing anything beyond ASCII
str_to_ints(unidecode.unidecode(s.lower()))

### Text splitting (tokenization)

We will cover three very common schemes:

- **character-level tokenization**: split on characters;
- **word-level tokenization**: split on spaces;
- **subword tokenization**: split based on frequency of character clusters (can be characters, parts of words, words, phrases)

Note also:
- **n-gram tokenization**: tokenize using the scheme of your choice, then create groups of length `n` ('abcd' -> 'a,b,c,d' -> 'ab,'bc','cd');
- **byte-level tokenization**: split on bytes.

You will note the use of the [`regex`](https://pypi.org/project/regex/) ("REGular EXpression") module (an updated version of the default [`re`](https://docs.python.org/3/library/re.html) package in Python). Regular expressions are *very* useful for any text manipulation: tutorials [here](https://realpython.com/regex-python/), [here](https://realpython.com/regex-python-part-2/) and [here](https://docs.python.org/3/howto/regex.html)), but they require some learning, and are not without pitfalls.

In [ ]:
s = "the quick brown fox jumps over the lazy dog"

In [ ]:
def split_chars(text):
    # return re.findall(r".", text)
    # an order of magnitude faster...
    return list(text)

In [ ]:
chars = split_chars("The quick brown fox jumped over the lazy dog.")
chars[:12]

In [ ]:
def split_words(text):
    # '\w' any "word character"
    # '+' appearing once or more
    # '|' or
    # [] any char inside the brackets (punctuation)
    return re.findall(rf"\w+|[{string.punctuation}]", text)

In [ ]:
# the punctuation characters we isolate in the regex
string.punctuation

In [ ]:
split_words("The quick brown fox jumped over the dog.")

### Vocabulary indexing

From a dataset, we create a big table **associating each word with a number**.

We can also add special information like an index for:
- **unknown words** ("\[UNK\]" for *unknown*, aka OOV: out of vocabulary);
- **beginning of sentence** ("bos");
- **end of sentence** ("eos");
- **padding** ("\[PAD\]", to make sequences the same length);
- **masked token**, and other customised "special tokens" depending on needs...

In [ ]:
vocabulary = {
    "[UNK]": 0,
    "the": 1,
    "quick": 2,
    "brown": 3,
    "fox": 4,
    "jumped": 5,
    "over": 6,
    "dog": 7,
    ".": 8,
}
words = split_words("The quick brown fox jumped over the lazy dog.")
indices = [vocabulary.get(word, 0) for word in words]

In [ ]:
indices

In [ ]:
reverse_vocab = {v:k for k,v in vocabulary.items()}

print(" ".join([reverse_vocab[i] for i in indices]))

The full pipeline:

<!-- <img style="height: 400px" src="images/chollet/figure14.1.png"> -->
<img style="height: 400px" src="https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/07/images/chollet/figure14.1.png">

[DLWP](https://deeplearningwithpython.io/chapters/chapter14_text-classification/#preparing-text-data), Figure 14.1

### Types of tokenzation

#### Char tokenization

In this first example, we'll isolate every character in the text, and map that to one integer.

In [ ]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        # an order of magnitude faster than using regex...
        return list(inputs)
        # return re.findall(r".", inputs)

    def index(self, tokens):
        # translate back into strings using our vocab, with '[UNK]' as fallback
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        # the standardization is to simplify things, but you could remove it as well
        inputs = self.standardize(inputs)
        # split into characters
        tokens = self.split(inputs)
        # convert from characters to integers
        indices = self.index(tokens)
        return indices

In [ ]:
def compute_char_vocabulary(inputs, max_size):
    char_counts = collections.Counter()
    for x in inputs:
        # same optional standardization step as before
        x = x.lower()
        # an order of magnitude faster than using regex...
        tokens = list(x)
        # tokens = re.findall(r".", x)
        char_counts.update(tokens)
    vocabulary = ["[UNK]"]
    # get the most common minus 1 (we already have '[UNK]')
    most_common = char_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)
    return dict((token, i) for i, token in enumerate(vocabulary))

#### Word tokenization

Then, if we use the same splitting method as before (on words – `\w+`, "one or more word character" in regex –, or any punctuation character), our vocabulary will be much larger, but the resulting encoding will be shorter

In [ ]:
class WordTokenizer:
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        # split into either words or punctuation signs ('[]' means match 'any of the characters inside')
        return re.findall(rf"\w+|[{string.punctuation}]", inputs)

    def index(self, tokens):
        # translate back into strings using our vocab, with '[UNK]' as fallback
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        # the standardization is to simplify things, but you could remove it as well
        inputs = self.standardize(inputs)
        # split into words (and/or punct)
        tokens = self.split(inputs)
        # convert tokens into integers
        indices = self.index(tokens)
        return indices

In [ ]:
def compute_word_vocabulary(inputs, max_size):
    word_counts = collections.Counter()
    for x in inputs:
        # same optional standardization step as before
        x = x.lower()
        # split into either words or punctuation signs ('[]' means match 'any of the characters inside')
        tokens = re.findall(rf"\w+|[{string.punctuation}]", x)
        word_counts.update(tokens)
    vocabulary = ["[UNK]"]
    if max_size > 0:
        # get the most common minus 1 (we already have '[UNK]')
        most_common = word_counts.most_common(max_size - len(vocabulary))
    else:
        # for max_size == 0, get everything
        most_common = word_counts.most_common()
    for token, count in most_common:
        vocabulary.append(token)
    return dict((token, i) for i, token in enumerate(vocabulary))

#### Test our tokenizers so far

Here we use Melville's [Moby Dick](https://www.gutenberg.org/ebooks/2701), but feel free to try other texts as well!

[`keras.utils.get_file` documentation](https://keras.io/api/utils/python_utils/#getfile-function)

In [ ]:
filename = keras.utils.get_file(
    origin="https://www.gutenberg.org/cache/epub/2701/pg2701.txt",
)

with open(filename, "r") as i:
    # this automatically splits into lines
    moby_dick = list(i)

##### Char tokenizer

Small vocabulary, longer encoding!

In [ ]:
vocabulary = compute_char_vocabulary(moby_dick, max_size=100)
char_tokenizer = CharTokenizer(vocabulary)

In [ ]:
print("Vocabulary length:", len(vocabulary))

In [ ]:
print("Vocabulary start:", list(vocabulary.keys())[:10])

In [ ]:
print("Vocabulary end:", list(vocabulary.keys())[-10:])

In [ ]:
print("Line length:", len(char_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

##### Word tokenizer

Large vocabulary, but shorter encoding!

In [ ]:
# you could try max_size=0: even larger vocab, although for the example sentence below it wouldn't change anything
vocabulary = compute_word_vocabulary(moby_dick, max_size=2_000)
word_tokenizer = WordTokenizer(vocabulary)

In [ ]:
print("Vocabulary length:", len(vocabulary))

In [ ]:
print("Vocabulary start:", list(vocabulary.keys())[:5])

In [ ]:
print("Vocabulary end:", list(vocabulary.keys())[-5:])

In [ ]:
print("Line length:", len(word_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

#### Subword tokenization

In [ ]:
data = [
    "the quick brown fox",
    "the slow brown fox",
    "the quick brown foxhound",
]

In [ ]:
def count_and_split_words(data, apply_standardization=False):
    counts = collections.Counter()
    for line in data:
        # in production system, no standardization is applied!
        if apply_standardization:
            line = line.lower()
        # for each match (a word or a punctuation sign)
        for match in re.finditer(rf"\w+|[{string.punctuation}]", line):
            word = match.group()
            start = match.start()
            # ignore the beginning of the string, then if char at start is preceded by space, create prefix
            prefix = "Ġ" if start > 0 and line[start - 1].isspace() else ""
            # decompose into chars
            # chars = re.findall(r".", word)
            # decompose into chars (list(word)), but include the prefix with the first char
            chars = ([prefix + word[0]] + list(word[1:])) if word else []
            # rejoin into a string with spaces between
            split_word = " ".join(chars)
            # add the split word to the counter
            counts[split_word] += 1

            # print(f"{match=} | {word=} | {chars=}")

    return dict(counts)

counts = count_and_split_words(data)

In [ ]:
counts

In [ ]:
def count_pairs(counts):
    # create a counter for pairs of symbols
    pairs = collections.Counter()
    # for all words
    for word, freq in counts.items():
        # split on space -> symbols (chars or char groups)
        symbols = word.split()
        # each time a pair of symbols appears, add the frequency of the word it appears in
        for pair in zip(symbols[:-1], symbols[1:]):
            pairs[pair] += freq
    return pairs

def merge_pair(counts, first, second):
    # regex:
    # - (?<!A)B find B if not (!) preceded (<) by A
    # - A(?!B) find A if not (!) preceding B
    # for first ("ab", second ("cd"), will match a string
    # with [any space]ab cd[any space], so
    # '... ab cd ...', but not '...ab cd...'
    # (the double \\ is required to escape – make valid – the \)
    split = re.compile(fr"(?<!\S){first} {second}(?!\S)")
    merged = f"{first}{second}"
    # we find 'X ab cd Y', and substitute it with 'X abcd Y', in all words
    # pattern.sub(replacement, target string), https://docs.python.org/3/library/re.html#re.Pattern.sub
    return {
        split.sub(merged, word): count
        for word, count in counts.items()
    }

In [ ]:
# the number 5 is arbitrary:
counts_bpe = counts.copy()
for i in range(5):

    pairs = count_pairs(counts_bpe)
    # first, second = max(pairs, key=pairs.get)
    [(first, second), freq] = pairs.most_common(n=1)[0]

    print(f"{i+1} | merging: {first} {second} → {first}{second} ({freq} occurrences)")
    print()
    print(f"{list(counts_bpe.keys())}")

    counts_bpe = merge_pair(counts_bpe, first, second)

    print(f"{list(counts_bpe.keys())}")
    print()
    print("-" * 20)
    print()

In [ ]:
def compute_sub_word_vocabulary(dataset, n_merges, apply_standardization=True):
    counts = count_and_split_words(dataset, apply_standardization=apply_standardization)

    char_counts = collections.Counter()
    for word in counts:
        for char in word.split():
            char_counts[char] += counts[word]
    most_common = char_counts.most_common()
    # add the unknown token at the star
    vocab = ["[UNK]"] + [char for char, freq in most_common]
    v_l = len(vocab)
    print(f"vocab size before merging: {v_l}")

    merges = []

    # continue merging until we have either merged everything or reached our vocab size
    while len(vocab) < n_merges:
        pairs = count_pairs(counts)
        if not pairs:
            break
        # find the most frequent pair
        first, second = max(pairs, key=pairs.get)
        # merge the pair in all words
        counts = merge_pair(counts, first, second)
        # append the merged pair as token,
        vocab.append(f"{first}{second}")
        # and the merge itself (we need that to decode!)
        merges.append(f"{first} {second}")

        clear_output(wait=True)
        print(f"vocab size before merging: {v_l}")
        print(f"{len(vocab)}/{n_merges}")
    else:
        print(f"vocab size now: {len(vocab)}")

    vocab = dict((token, index) for index, token in enumerate(vocab))
    # note the 'rank': the most frequent merge comes first in our list
    merges = dict((token, rank) for rank, token in enumerate(merges))
    return vocab, merges

In [ ]:
apply_standardization=False
vocabulary, merges = compute_sub_word_vocabulary(moby_dick, 2000, apply_standardization=apply_standardization)

In [ ]:
print("Vocabulary length:", len(vocabulary))
print("Vocabulary start:", list(vocabulary.keys())[:10])
print("Vocabulary end:", list(vocabulary.keys())[-7:])

In [ ]:
print("Merges length:", len(merges))
print("Merges start:", list(merges.keys())[:10])
print("Merges end:", list(merges.keys())[-7:])

In [ ]:
class SubWordTokenizer:
    def __init__(self, vocabulary, merges, apply_standardization=True):
        self.vocabulary = vocabulary
        self.invert_vocab = {v:k for k, v in self.vocabulary.items()}
        self.merges = merges
        self.unk_id = vocabulary["[UNK]"]
        self.apply_standardization = apply_standardization

    def standardize(self, inputs):
        return inputs.lower()

    def bpe_merge(self, word):
        while True:
            # Matches all symbol pairs in the text
            pairs = re.findall(r"(?<!\S)\S+ \S+(?!\S)", word, overlapped=True)
            if not pairs:
                break
            # We apply merge rules in "rank" order. More frequent pairs
            # are merged first → they have the lowest rank
            best = min(pairs, key=lambda pair: self.merges.get(pair, 1e9))
            if best not in self.merges:
                break
            first, second = best.split()
            split = re.compile(rf"(?<!\S){first} {second}(?!\S)")
            merged = f"{first}{second}"
            word = split.sub(merged, word)
        return word

    def split(self, inputs):
        tokens = []
        # Split words
        for match in re.finditer(rf"\w+|[{string.punctuation}]", inputs):
            word = match.group()
            start = match.start()
            # ignore the beginning of string, then: if first match char is preceded by space, create prefix
            prefix = "Ġ" if start > 0 and inputs[start - 1].isspace() else ""
            # decompose into chars (list(word)), but include the prefix with the first char
            word = " ".join([prefix + word[0]] + list(word[1:]))
            # Applies byte-pair encoding merge rules
            word = self.bpe_merge(word)
            tokens.extend(word.split())
        return tokens

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        if self.apply_standardization:
            inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

    def detokenize(self, token_ids):
        # we assume here that "Ġ" only appears as a substitute of space...
        return "".join(self.invert_vocab[i] for i in token_ids).replace("Ġ", " ")

In [ ]:
sub_word_tokenizer = SubWordTokenizer(vocabulary, merges, apply_standardization=apply_standardization)

In [ ]:
tokenized_s = sub_word_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)
print(tokenized_s)
print("Line length:", len(tokenized_s))

In [ ]:
sub_word_tokenizer.detokenize(tokenized_s)

Note that an important piece of the pipeline is missing: in production versions of BPE, the tokenizer computes the merges on raw bytes, and always includes, as a 'backup', the minimal 256 tokens vocabulary (all possibilities for one byte), which means that in the worst case, it will always be able to encode/decode sequences without resorting to '[UNK]'. See the reference notebook for ressources to learn more about this!

---

## Sets vs sequences

- Sets: "bag of words", the order is lost.  
- Sequences: order/time matters!

### Preparing the IMDB movie reviews data

#### Downloading the data & organise the directory

In [ ]:
DATASET_DIR = pathlib.Path("aclImdb")

if not DATASET_DIR.exists():
    !curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
    !tar -xf aclImdb_v1.tar.gz # this untars the archive to a folder called aclImdb
    !rm -r aclImdb/train/unsup

MODELS_DIR = pathlib.Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# to read a review:
# !cat aclImdb/train/pos/4077_10.txt

In [ ]:
# code to split the data into train/val folders

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"
for category in ("neg", "pos"):
    # do this only once
    if not os.path.isdir(VAL_DIR / category):
        # make 'neg'/'pos' dir in validation
        os.makedirs(VAL_DIR / category)
        # list files in 'train'
        files = os.listdir(TRAIN_DIR / category)
        # shuffle using a seed
        random.Random(1337).shuffle(files)
        # 2% of our samples for validation
        num_val_samples = int(0.2 * len(files))
        val_files = files[-num_val_samples:]
        # move our files
        for fname in val_files:
            shutil.move(TRAIN_DIR / category / fname,
                        VAL_DIR / category / fname)

#### Process using `text_dataset_from_directory`

See the [documentation](https://keras.io/api/data_loading/text/) (similar Keras helper functions to create a [Tensorflow Dataset](https://www.tensorflow.org/api_docs/python/tf/data/Dataset) as `image_dataset_from_directory` we've seen).


```bash
$ tree -d 1 aclImdb
├── aclImdb
│   ├── test
│   │   ├── neg
│   │   └── pos
│   ├── train
│   │   ├── neg
│   │   └── pos
│   └── validation
│       ├── neg
│       └── pos
```

In [ ]:
BATCH_SIZE = 32

# each of these iterables returns tuples containing two tensors:
# samples, shape: (batch_size, sample_shape) ← our texts
# targets, shape: (batch_size,)              ← 0 or 1
train_ds = keras.utils.text_dataset_from_directory(
    TRAIN_DIR, batch_size=BATCH_SIZE
)
val_ds = keras.utils.text_dataset_from_directory(
    VAL_DIR, batch_size=BATCH_SIZE
)
test_ds = keras.utils.text_dataset_from_directory(
    TEST_DIR, batch_size=BATCH_SIZE
)

##### Test

In [ ]:
# displaying the shapes and dtypes of the first batch
for inputs, targets in train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print()
    print("targets[0]:", targets[0].numpy())
    print()
    print("inputs[0]:", inputs[0].numpy())
    break

### Baseline

Do you remember the baseline for the IMDB dataset?

If not, you can find it in [`lectures/03/chapter04_classification-and-regression.1.imdb.ipynb`](https://github.com/jchwenger/DLWP/blob/main/lectures/03/chapter04_classification-and-regression.1.imdb.ipynb)

---

## Set models

### Training a bag-of-words model

#### Bag-of-words (unigrams), multi-hot encoding

- bag-of-word (unigram): first you split, then take each split on its own
- binary encoding: multi-hot
- the order is lost

The same set-up as when dealing with IMDB before (we did it manually then).

In [ ]:
max_tokens = 20_000
# preprocessing our datasets with a `TextVectorization` layer
text_vectorization = keras.layers.TextVectorization(
    # cap the vocab size
    max_tokens=max_tokens,
    # text splitting (the default)
    split="whitespace",
    # Outputs a integer sequence of token IDs
    output_mode="multi_hot",
    # the default (↔ unigram)
    ngrams=None,
)

# we must build our text encoding layer to the text at hand (build the vocab, etc.)
# for that, we 1) create a sub-dataset that returns only the text
train_ds_no_labels = train_ds.map(lambda x, y: x)
# then 2) we use `.adapt` on it
text_vectorization.adapt(train_ds_no_labels)

bag_of_words_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bag_of_words_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bag_of_words_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

##### Tests

In [ ]:
# unigram == bag of words (BOW)
set("the cat sat on the mat".split())

In [ ]:
vocabulary = text_vectorization.get_vocabulary()
inverse_vocab = dict(enumerate(vocabulary))

We get a multi-hot vector, with `1` at the index of the words present.

In [ ]:
cat = text_vectorization("the cat")
# "the cat" is now a vector of 20000 zeros  with 1s at the index of "the" and "cat"
print(cat.numpy(), cat.shape)
print()
# retrieving the indices of the words
indz = tf.where(cat > 0)
print(indz.numpy())
print()
# retranslating back into a string
print(" ".join([inverse_vocab[i.numpy().item()] for i in indz]))

In [ ]:
vocabulary.index("cat"), inverse_vocab[vocabulary.index("cat")]

In [ ]:
cat = text_vectorization("the cute cat")
print(cat.numpy(), cat.shape)
print()
indz = tf.where(cat > 0)
print(indz.numpy())
print()
print(" ".join([inverse_vocab[i.numpy().item()] for i in indz]))

In [ ]:
vocabulary.index("cute"), inverse_vocab[vocabulary.index("cute")]

In [ ]:
# inspecting the output of our binary unigram dataset
for inputs, targets in bag_of_words_train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print()
    print("targets[0]:", targets[0].numpy())
    print()
    print("inputs[0]:", inputs[0].numpy())
    break

#### Model, training

In [ ]:
def build_linear_classifier(max_tokens, name, clear=True):
    if clear:
        keras.backend.clear_session()
    inputs = keras.Input(shape=(max_tokens,))
    outputs = keras.layers.Dense(1, activation="sigmoid")(inputs)
    model = keras.Model(inputs, outputs, name=name)
    model.compile(
        optimizer="adam",
        # our labels are only 0 or 1 (negative/positive)
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
# Training the binary unigram model
model = build_linear_classifier(max_tokens, "bag_of_words_classifier")
model.summary(line_length=80)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)

history = model.fit(
    bag_of_words_train_ds,
    validation_data=bag_of_words_val_ds,
    epochs=10,
    callbacks=[
        early_stopping, 
        keras.callbacks.ModelCheckpoint(
            MODELS_DIR / "bow_classifier.keras",
            save_best_only=True
        )
    ],
    verbose=0,
)

In [ ]:
plot_history(history)

In [ ]:
# And testing it
_, test_acc = model.evaluate(bag_of_words_test_ds)
print(f"Test acc: {test_acc:.3f}")

In [ ]:
del (
    model, text_vectorization,
    bag_of_words_train_ds, bag_of_words_val_ds, bag_of_words_test_ds
)
keras.backend.clear_session()

### Training a bigram model

#### Bigrams (splitting → grouping by two), multi-hot encoding

- bigram: first you split, then take tokens two by two   
- binary encoding: multi-hot (each bigram is one token!)
- *almost all* order is lost

In [ ]:
max_tokens = 20_000
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    # the default
    split="whitespace",
    output_mode="multi_hot",
    # returns bigrams
    ngrams=2,
)

# reusing text only ds from `Bag-of-words (unigrams), multi-hot encoding`
text_vectorization.adapt(text_only_train_ds)

bigram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
bigram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
bigram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

##### Tests

In [ ]:
vocabulary = text_vectorization.get_vocabulary()
print(f"{len(vocabulary)} tokens in the vocabulary")

# displaying the vocabulary: note the groups of two words!
print([x.item() for x in vocabulary[100:108]])

In [ ]:
encoded_sentence = text_vectorization("I write, rewrite, and still rewrite again")
print(encoded_sentence)

#### Model, training

In [ ]:
# Training the binary bigram model
model = build_linear_classifier(max_tokens, "bigram_classifier")

history = model.fit(
    bigram_train_ds,
    validation_data=bigram_val_ds,
    epochs=10,
    callbacks=[
        early_stopping, 
        keras.callbacks.ModelCheckpoint(
            MODELS_DIR / "bigram_classifier.keras",
            save_best_only=True
        )
    ],
    verbose=0,
)

In [ ]:
plot_history(history)

In [ ]:
#  and testing
_, test_acc = model.evaluate(bigram_test_ds)
# better than the unigram model
print(f"Test acc: {test_acc:.3f}")

In [ ]:
del (
    model, text_vectorization,
    bigram_train_ds, bigram_val_ds, bigram_test_ds
)
keras.backend.clear_session()

## Sequence models

In [ ]:
# Preparing integer sequence train/val/test datasets
# we cut our sequences to 600 words max, and max tokens to 8000! (For memory.)
# This will affect performance...
max_length = 600
max_tokens = 30_000

text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(train_ds_no_labels)

sequence_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),num_parallel_calls=8
)
sequence_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),num_parallel_calls=8
)
sequence_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),num_parallel_calls=8
)

##### Tests

In [ ]:
x, y = next(sequence_test_ds.as_numpy_iterator())
print(x.shape)
print(x)

#### Training a recurrent model with `one_hot` vectors


In [ ]:
# A sequence model built on one-hot encoded vector sequences
hidden_dim = 64
inputs = keras.Input(shape=(None,), dtype="int64")
# ↓ our one-hot vectors ---------------------------------------
embedded = keras.ops.one_hot(inputs, max_tokens)
#------------------------------------------------ passed here ↓
x = keras.layers.Bidirectional(keras.layers.LSTM(hidden_dim))(embedded)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# ((max_tokens + hidden_dim + 1) * 64) * 4 * 2 params in the LSTM layer (* 2 since bidirectional...)
model.summary()

In [ ]:
history = model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[
        early_stopping, 
        keras.callbacks.ModelCheckpoint(
            MODELS_DIR / "lstm_one_hot.keras",
            save_best_only=True
        )
    ],
    verbose=0,
)

In [ ]:
plot_history(history)

In [ ]:
model = keras.models.load_model(MODELS_DIR / "lstm_one_hot.keras")
print(f"Test acc: {model.evaluate(sequence_test_ds, verbose=0)[1]:.3f}")
del model

In [ ]:
del model
keras.backend.clear_session()

### Word embeddings

A big revolution of this last decade: researchers discovered that we can get systems to **learn to project words** into a **vector space** that retain **semantic relationship**.

The idea behind this discovery is that **similar words** (similar meanings) occur in **similar contexts**.

*You shall know a word by the company it keeps.*  
(J. R. Firth, "A Synopsis of Linguistic Theory", 1957, cf. also [the late Wittgenstein](https://plato.stanford.edu/entries/wittgenstein/#MeanUse))

The algorithms will train on large text corpora, and:
- count all the occurrences of words and contexts;  
- try and predict either the word given the context;
- or the context given the word.

Word embeddings in the context of a large dictionary have typically 256, 512 or 1024 dimensions  
(the dimension of a vector is its length i.e. the number of components).

One-hot encoded vectors can exceed 20,000 dimensions!  
Also, technically all one-hot encoded are **orthogonal**: no similarity between them.

Word embeddings compress the information into fewer dimensions.  
Two word vectors can be compared to each other!

<!-- <img style="height: 400px" src="images/chollet/figure14.3.png"> -->
<img style="height: 400px" src="https://github.com/jchwenger/DLWP/blob/main/lectures/07/images/chollet/figure14.3.png?raw=true">

[DLWP](https://deeplearningwithpython.io/chapters/chapter14_text-classification/#understanding-word-embeddings), Figure 14.3

|One-hot|Word embeddings|
|:---|:---|
|binary (integers: 0/1)|floating point vectors|
|sparse (most elements are zeros)|*dense*|
|very high-dimensional|low-to-medium-dimensional|
|hard-coded|learnt from data|


The results is that each token will be represented as a **coordinate** (aka a **vector**) in a high-dimensional space.

The most striking features of these spaces is that they seem to encode **semantic relationships**!

<!-- <img src="images/nlp/linear-relationships.svg"> -->
<img src="https://github.com/jchwenger/DLWP/blob/main/lectures/07/images/nlp/linear-relationships.svg?raw=true">


<small>[Embeddings: Translating to a Lower-Dimensional Space, Google Foundational Courses, Machine Learning, Embeddings](https://developers.google.com/machine-learning/crash-course/embeddings/translating-to-a-lower-dimensional-space)</small>

#### Note: universal embedding

A universal embedding is unlikely, or very difficult to achieve (although [recent work](https://phillipi.github.io/prh/) proposes the opposite conjecture, thanks Peyton Hammersley for the references).

Semantic relationships depend on task – the text corpus and what we are learning.

Expect different geometries for different tasks (e.g. sentiment analysis is very different from classification of legal documents).

#### Also: bias

The biases of your dataset **will be encoded** in the space (for instance, gendered associations between professions).

### Using a word embedding

Later, researchers discovered that you can simply **learn** these vectors with your DL model using backprop like everything else!

In current models, you just invoke a specific layer, and all the work is done for you.

Note that you must specify in advance the **dimensionality** of the embedding space.

As usual, more dimensions == more **resolution** (finer-grained), but more computationally expensive.

<!-- <img style="height: 400px" src="images/chollet/figure14.5.png"> -->
<img style="height: 400px" src="https://github.com/jchwenger/DLWP/blob/main/lectures/07/images/chollet/figure14.5.png?raw=true">

[DLWP](https://deeplearningwithpython.io/chapters/chapter14_text-classification/#using-a-word-embedding), Figure 14.5

The two arguments of the embedding layer code are:

```python
keras.layers.Embedding(
    input_dim=max_tokens,
    output_dim=hidden_dim
)
```

- input_dim = the size of our vocab
- output_dim = the dimension of the embedding space (each word vector)

In [ ]:
hidden_dim = 64
inputs = keras.Input(shape=(max_length,), dtype="int32")

# EMBEDDING LAYER
x = keras.layers.Embedding(
    # the size of the vocabulary
    input_dim=max_tokens,
    # the dimensionality of the embedding space
    output_dim=hidden_dim,
    # respect padding
    mask_zero=True,
)(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(hidden_dim))(x)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_embedding")
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

Note that given our vocabulary the embedding layer is **large**.

In [ ]:
model.summary(line_length=80)

In [ ]:
history = model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[
        early_stopping, 
        keras.callbacks.ModelCheckpoint(
            MODELS_DIR / "lstm_with_embedding.keras",
            save_best_only=True
        )
    ],
    verbose=0,
)

In [ ]:
plot_history(history)

In [ ]:
model = keras.models.load_model(MODELS_DIR / "lstm_with_embedding.keras")
print(f"Test acc: {model.evaluate(sequence_test_ds, verbose=0)[1]:.3f}")

In [ ]:
del model
keras.backend.clear_session()

#### Embedding layer: \# of parameters and dimensions

How many learnable parameters does the embedding weight matrix have?

$$
\bbox[5px,border:2px solid red]
{
\mathrm{input\_dim} \times \mathrm{output\_dim}
}
$$

That is:

$$
\bbox[5px,border:2px solid red]
{
\mathrm{vocab\_size} \times \mathrm{embed\_size}
}
$$

Example:

$ 10'000 \times 8 = 80'000$ elements.



The embedding layer takes as input tensors of shape `(batch_size, sequence_length)`.  
The outputs tensors of shape `(bach_size, sequence_length, output_dim)`.

As usual in Keras, the batch_size is represented as `None`:

$$
\bbox[5px,border:2px solid red]
{
In: (None, sequence\_length) \to Out: (None, sequence\_length, output\_dim)
}
$$

`output_dim` could be called `embed_dim`, the number of dimensions of our embedding space!



### Pretraining a word embedding

We will now build a model focussing only on learning embeddings, or useful token representations.

This model is called **Continuous Bag-of-Word (CBOW)** (Mikolov et al, [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781)).

The model is simply an embedding layer, then a dense layer trying to predict the probability of words (classification task) given a context.

<!-- <img style="height: 400px" src="images/chollet/figure14.6.png"> -->
<img style="height: 400px" src="https://github.com/jchwenger/DLWP/blob/main/lectures/07/images/chollet/figure14.6.png?raw=true">

[DLWP](https://deeplearningwithpython.io/chapters/chapter14_text-classification/#pretraining-a-word-embedding), Figure 14.6

In [ ]:
max_tokens = 30_000

tokenize_no_padding = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="int",
)

# reusing text only ds from `Bag-of-words (unigrams), multi-hot encoding`
tokenize_no_padding.adapt(train_ds_no_labels)

# Words to the left or right of label
context_size = 4
# Total window size
window_size = context_size * 2 + 1

def window_data(token_ids):
    # compute the length of the current sequence
    # the index of the last window is length - (window_size - 1)
    num_windows = tf.maximum(tf.size(token_ids) - context_size * 2, 0)
    # create indices [[0,1,...,window_size - 1]]
    windows = tf.range(window_size)[None, :]
    # use broadcasting to create sliding indices:
    # [[0,1,...,window_size - 1]] + [[0],
    #                                [1],
    #                                ...,
    #                                [window_size - 1]]
    # →
    # [[0,1,...,window_size - 1 + 0],
    #  [1,2,...,window_size - 1 + 1],
    #  [...,                   ...,]]
    windows = windows + tf.range(num_windows)[:, None]
    # use `gather` to retrieve the sequences given the indices: https://www.tensorflow.org/api_docs/python/tf/gather
    windowed_tokens = tf.gather(token_ids, windows)
    # create a new dataset using those
    return tf.data.Dataset.from_tensor_slices(windowed_tokens)

def split_label(window):
    left = window[:context_size]
    right = window[context_size + 1 :]
    bag = tf.concat((left, right), axis=0)
    label = window[context_size]
    return bag, label

# load the dataset, keep only the text
cbow_dataset = keras.utils.text_dataset_from_directory(TRAIN_DIR, batch_size=None)
cbow_dataset = cbow_dataset.map(lambda x, y: x, num_parallel_calls=8)

# Use the text-only dataset for tokenization
cbow_dataset_tokenized = cbow_dataset.map(tokenize_no_padding, num_parallel_calls=8)
# Creates context windows
cbow_dataset = cbow_dataset_tokenized.interleave(window_data, cycle_length=8, num_parallel_calls=8)
# Splits middle wonder into a label
cbow_dataset = cbow_dataset.map(split_label, num_parallel_calls=8)
cbow_dataset = cbow_dataset.batch(1024).cache()

#### Tests

##### `window_data`

In [ ]:
tf.range(10)[None, :]

In [ ]:
tf.range(20)[:, None]

In [ ]:
tf.range(10)[None, :] + tf.range(20)[:, None]

##### Dataset

In [ ]:
for bag, label in cbow_dataset.take(1):
    bag, label = bag[0], label[0]
    print(bag)
    print(label)

#### Baseline

In [ ]:
random_guess_prob = 1/max_tokens
print(f"A random guess would yield a probability of {random_guess_prob:.5f} for each token -> accuracy: {random_guess_prob * 100:.3f}%.")

In [ ]:
counts = tf.math.bincount(
    tf.concat([x for x in cbow_dataset_tokenized], axis=0),
    minlength=max_tokens
)
probs = counts / tf.reduce_sum(counts)

likeliest_class = tf.math.argmax(probs).numpy().item()
prob_likeliest_class = probs[likeliest_class].numpy().item()
print(f"Likeliest class: {likeliest_class}, probability: {prob_likeliest_class} -> accuracy: {prob_likeliest_class * 100:.2f}%")

In [ ]:
loss = -tf.reduce_sum(probs * tf.math.log(probs + 1e-9))
print(f"Entropy of the dataset: {loss.numpy().item()}.")

#### Model, training

In [ ]:
hidden_dim = 64
inputs = keras.Input(shape=(2 * context_size,))
cbow_embedding = keras.layers.Embedding(
    max_tokens,
    hidden_dim,
)
x = cbow_embedding(inputs)
x = keras.layers.GlobalAveragePooling1D()(x)
outputs = keras.layers.Dense(max_tokens, activation="softmax")(x)
cbow_model = keras.Model(inputs, outputs)
cbow_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)

In [ ]:
cbow_model.fit(
    cbow_dataset,
    epochs=4,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            MODELS_DIR / "cbow_embeddings.keras",
            save_best_only=True,
        ),    
    ],
)

In [ ]:
del cbow_dataset

### Using the pretrained embedding for classification

Pretrained word embeddings are useful **when training data is limited** – just as with pretrained convnets.

Very structured embeddings hopefully capture **generic structure** appropriate to diverse domains.

(The more data you can train on, the more likely your task-specific embeddings will perform better.)

In [ ]:
inputs = keras.Input(shape=(max_length,))
lstm_embedding = keras.layers.Embedding(
    input_dim=max_tokens,
    output_dim=hidden_dim,
    mask_zero=True,
)
x = lstm_embedding(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(hidden_dim))(x)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_cbow")

In [ ]:
# use the embeddings of our trained model
lstm_embedding.embeddings.assign(cbow_embedding.embeddings)

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[
        early_stopping,
        keras.callbacks.ModelCheckpoint(
            MODELS_DIR / "lstm_with_cbow.keras",
            save_best_only=True
        ),        
    verbose=0,
)

In [ ]:
print_history(history)

In [ ]:
model = keras.models.load_model(MODELS_DIR / "lstm_with_cbow.keras")
print(f"Test acc: {model.evaluate(sequence_test_ds, verbose=0)[1]:.3f}")

More about pre-trained embedding, and vector arithmetic, in [`lectures/07.more/2nd-ed.chapter11_part02_sequence-models_glove.ipynb`](https://github.com/jchwenger/DLWP/blob/main/lectures/07.more/2nd-ed.chapter11_part02_sequence-models_glove.ipynb).

### Save models to Google Drive


In [ ]:
EXPORT=False

if EXPORT:
    # zip models
    !zip sequences.models.zip {MODELS_DIR}/*
    # connect to drive
    from google.colab import drive
    drive.mount('/content/drive')
    # copy zip to drive (adjust folder as needed)
    !cp sequences.models.zip drive/MyDrive/gold/IS53024B-Artificial-Intelligence/models

---

## Summary

### NLP

- **standardisation** / **tokenisation** / **indexing**
- tokenisation: various possible **levels** (words, characters, n-grams, etc.)

- **Two kinds of NLP models**:
  - **bag-of-words** sets of words or n-grams, the order is lost.
  - **sequence models** that process word order.

### Word embbeddings

- **Various kinds of word encodings**:
  - **one-hot/multi-hot**: the presence of words is marked by a 1 (binary) → *sparse* & *hard-coded*
  - **word embeddings**: project words/tokens into vector spaces where collocations between words ("the company a word keeps") are modeled as the distance between vectors. → *dense* & *learnt from data*
- **Embedding layers** can be trained end to end with your net!
- The **Continuous Bag of Word (CBOW)** model learns to predict one token given a context window.
- **Pretrained embeddings** can also be used on top of your own models, like pretrained networks!
  